In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from collections import defaultdict


# --- Directory Configuration ---
# CHANGED: Updated base_path as requested
base_path = r"./dataset/"

folders_path = [
    os.path.join(base_path, "3-Phase-current-0.7mm-bearing-fault"),
    os.path.join(base_path, "3-Phase-current-0.9mm-bearing-fault"),
    os.path.join(base_path, "3-Phase-current-1.1mm-bearing-fault"),
    os.path.join(base_path, "3-Phase-current-1.3mm-bearing-fault"),
    os.path.join(base_path, "3-Phase-current-1.5mm-bearing-fault"),
    os.path.join(base_path, "3-Phase-current-1.7mm-bearing-fault"),
    os.path.join(base_path, "3-Phase-current-12-4mm-broken-rotor-bar-fault"),
    os.path.join(base_path, "3-Phase-current-healthy-motor")
]

fault_mapping = {
    "3-Phase-current-healthy-motor": "Healthy",
    "3-Phase-current-0.7mm-bearing-fault": "Unhealthy - 0.7mm Bearing Fault",
    "3-Phase-current-0.9mm-bearing-fault": "Unhealthy - 0.9mm Bearing Fault",
    "3-Phase-current-1.1mm-bearing-fault": "Unhealthy - 1.1mm Bearing Fault",
    "3-Phase-current-1.3mm-bearing-fault": "Unhealthy - 1.3mm Bearing Fault",
    "3-Phase-current-1.5mm-bearing-fault": "Unhealthy - 1.5mm Bearing Fault",
    "3-Phase-current-1.7mm-bearing-fault": "Unhealthy - 1.7mm Bearing Fault",
    "3-Phase-current-12-4mm-broken-rotor-bar-fault": "Unhealthy - Broken Rotor Bar Fault"
}


# --- Configuration Parameters (from original script) ---
num_rows_per_segment = 100
num_cols = 1000
required_values_per_segment = num_rows_per_segment * num_cols # 100,000
current_cols = [' Current-A', ' Current-B', ' Current-C']
column_names = [f'Feature {i+1}' for i in range(num_cols)]


# --- Global lists to track issues ---
missing_files = []
insufficient_data_files = []
missing_column_files = []


# --- Function to process a single file and apply label (Modified from original) ---
def process_file_for_labeling(filename, label):
    """
    Loads a file, reshapes Current A, B, and C into 100x1000 segments
    using 100,000 sequential values from each, concatenates them to 300x1000,
    and adds a status label.
    """
    try:
        # Use low_memory=False for large files
        # The filename here will be the absolute path found by glob.glob
        df = pd.read_csv(filename, low_memory=False)
    except FileNotFoundError:
        global missing_files
        missing_files.append(filename)
        return None
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return None

    segments = []

    # Check data sufficiency for all three current columns
    for col_name in current_cols:
        # Check for column existence and clean up column names by stripping whitespace
        # This modification helps if the column names in the CSV have leading/trailing spaces
        
        # Clean the column names in the DataFrame for reliable access
        df.columns = df.columns.str.strip()
        
        # Now check if the cleaned column name (without leading space) exists
        cleaned_col_name = col_name.strip()
        
        if cleaned_col_name not in df.columns:
            # If the specific column is not found
            global missing_column_files
            missing_column_files.append(f"{filename} (Missing column: {col_name})")
            return None

        data = df[cleaned_col_name].values # Use the cleaned column name

        # Take the first 100,000 sequential values
        data_segment = data[:required_values_per_segment]

        # CRITICAL CHECK: Ensure we have enough data points (100,000)
        if len(data_segment) < required_values_per_segment:
            global insufficient_data_files
            insufficient_data_files.append(f"{filename} (Only {len(data_segment)} values found for {cleaned_col_name}. Requires 100,000)")
            return None # Skip the entire file if any current column is too short

        # Reshape to 100 rows x 1000 columns
        reshaped_array = data_segment.reshape((num_rows_per_segment, num_cols))

        # Convert to DataFrame and assign feature names
        df_segment = pd.DataFrame(reshaped_array, columns=column_names)
        segments.append(df_segment)

    # Concatenate the three segments vertically (300 x 1000 data columns)
    final_df = pd.concat(segments, ignore_index=True)

    # Add the detailed Status column
    final_df['Status'] = label

    return final_df


# --- Dynamic File Discovery and Processing ---
all_processed_dfs = []
processed_file_count = 0
file_tracking_summary = defaultdict(int)


print("--- Starting Data Discovery and Processing ---")
print(f"Base Directory: {base_path}")
print("---")

for folder_path in folders_path:
    # Get the base folder name (e.g., "3-Phase-current-healthy-motor")
    folder_name = os.path.basename(folder_path)

    # Get the corresponding detailed label from the map
    label = fault_mapping.get(folder_name, "Unknown Fault")

    # Find all CSV files in the current folder
    file_list = glob.glob(os.path.join(folder_path, '*.csv'))

    print(f"\nProcessing Folder: '{folder_name}' ({len(file_list)} files found, Label: {label})")

    for filename in file_list:
        df_processed = process_file_for_labeling(filename, label)
        if df_processed is not None:
            all_processed_dfs.append(df_processed)
            processed_file_count += 1
            file_tracking_summary[label] += 1


# --- Merge all processed DataFrames ---
if all_processed_dfs:
    df_final_merged = pd.concat(all_processed_dfs, ignore_index=True)
else:
    print("\nError: No files were processed successfully.")
    # Report errors even if no files were processed
    if missing_files or insufficient_data_files or missing_column_files:
        print("\n--- WARNING: Issues found during processing: ---")
        if missing_files:
            print("\nFiles Listed but Not Found:")
            for mf in sorted(list(set(missing_files))):
                print(f"- {mf}")
        if insufficient_data_files:
            print("\nFiles Skipped Due to Insufficient Data:")
            for idf in sorted(list(set(insufficient_data_files))):
                print(f"- {idf}")
        if missing_column_files:
            print("\nFiles Skipped Due to Missing Current Columns:")
            for mcf in sorted(list(set(missing_column_files))):
                print(f"- {mcf}")
    # Since we are in an execution environment, we cannot exit/halt here, 
    # but the error reporting is complete.
    exit()

# NEW STEP: Shuffle the final merged DataFrame
print("\nShuffling the final merged dataset...")
# frac=1 ensures 100% of the data is returned (i.e., shuffled)
# reset_index(drop=True) discards the old index and creates a new sequential one
df_final_merged = df_final_merged.sample(frac=1, random_state=42).reset_index(drop=True)  


# --- Save the final merged DataFrame to a specific directory ---

# CHANGED: Updated output directory to reflect the new base path
output_dir = r"./processed_data" 
output_filename = 'shuffled_processed_output.csv' # Changed filename to reflect shuffling
full_output_path = os.path.join(output_dir, output_filename)

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)  

# Save the final merged DataFrame to the specified path
df_final_merged.to_csv(full_output_path, index=False)


print("\n" + "="*50)
print(f"Data processing and **shuffling** complete. {processed_file_count} files were successfully processed.")
print(f"Merged and **Shuffled** file saved as **'{full_output_path}'** with shape {df_final_merged.shape}.")
print("="*50)

print("\nFinal Status Counts (Total Rows: 300 rows per file):")

# Summarize by the detailed fault label
for label, count in file_tracking_summary.items():
    print(f"- {label}: {count} files processed ({count*300} rows)")
print("-" * 30)
print(df_final_merged['Status'].value_counts())

if missing_files or insufficient_data_files or missing_column_files:
    pass

--- Starting Data Discovery and Processing ---
Base Directory: C:\Users\MAHAD ENTERPRISES\OneDrive\Desktop\Current Signature Dataset of Three-Phase Induction Motor under Varying Load Conditions
---

Processing Folder: '3-Phase-current-0.7mm-bearing-fault' (6 files found, Label: Unhealthy - 0.7mm Bearing Fault)

Processing Folder: '3-Phase-current-0.9mm-bearing-fault' (6 files found, Label: Unhealthy - 0.9mm Bearing Fault)

Processing Folder: '3-Phase-current-1.1mm-bearing-fault' (6 files found, Label: Unhealthy - 1.1mm Bearing Fault)

Processing Folder: '3-Phase-current-1.3mm-bearing-fault' (6 files found, Label: Unhealthy - 1.3mm Bearing Fault)

Processing Folder: '3-Phase-current-1.5mm-bearing-fault' (6 files found, Label: Unhealthy - 1.5mm Bearing Fault)

Processing Folder: '3-Phase-current-1.7mm-bearing-fault' (6 files found, Label: Unhealthy - 1.7mm Bearing Fault)

Processing Folder: '3-Phase-current-12-4mm-broken-rotor-bar-fault' (2 files found, Label: Unhealthy - Broken Rotor Ba

In [5]:
# ====================================================================
# --- ADDED STEP: Load and Inspect the Saved Data ---
# ====================================================================

# The file was saved at full_output_path defined earlier in the script.
# We load it back into a new DataFrame named 'df'.
full_output_path = r"C:\Users\MAHAD ENTERPRISES\OneDrive\Desktop\Processed_Motor_Data\shuffled_processed_output.csv" 
output_filename = 'shuffled_processed_output.csv'

try:
    df = pd.read_csv(full_output_path)
    print(f"\nSuccessfully loaded '{output_filename}' into DataFrame 'df' for further analysis.")

    print("\nShape of loaded data")
    print(f"DataFrame shape: {df.shape}")
    print("\nFirst 5 rows of the loaded DataFrame:")
    print(df.head())
    
except FileNotFoundError:
    print(f"\nError: The file was not found at the specified path: {full_output_path}")
    print("Please ensure the data processing script has sufficient permissions.")
except Exception as e:
    print(f"\nAn error occurred while reading the CSV file: {e}")

# --- Next Step ---
# The processed data is now loaded and verified, ready for machine learning.
print("\n--- Next Step ---")
print("The processed data is ready for machine learning.")
print(f"The DataFrame 'df' (shape: {df.shape}) is available for tasks like k-NN.")
print("=" * 50)


Successfully loaded 'shuffled_processed_output.csv' into DataFrame 'df' for further analysis.

Shape of loaded data
DataFrame shape: (11400, 1001)

First 5 rows of the loaded DataFrame:
   Feature 1  Feature 2  Feature 3  Feature 4  Feature 5  Feature 6  \
0     2.0366     2.0476     2.0476     2.0476     2.0476     2.0476   
1     2.9963     2.9963     2.9963     2.9963     2.9963     2.9963   
2     1.8877     1.8877     1.8877     1.8877     1.8877     1.8877   
3     3.0049     3.0049     3.0049     3.0049     3.0049     3.0049   
4     2.8620     2.8620     2.8620     2.8620     2.8620     2.8620   

   Feature 7  Feature 8  Feature 9  Feature 10  ...  Feature 992  Feature 993  \
0     2.0476     2.0476     2.0476      2.0476  ...       2.0208       2.0208   
1     2.9963     2.9963     2.9963      2.9963  ...       1.9866       1.9866   
2     1.8877     1.8877     1.8877      1.8877  ...       2.9621       2.9621   
3     3.0049     3.0049     3.0049      3.0049  ...       2.88

In [6]:
import numpy as np
from collections import Counter

# ====================================================================
# --- CUSTOM K-NN MODEL IMPLEMENTATION WITHOUT BUILD-IN LIBRARIES ---
# ====================================================================

# --- Core: Euclidean Distance Function ---
def euclidean_distance(point1, point2):
    """
    Calculates the Euclidean distance between two data points (vectors).
    d(p, q) = sqrt(sum((p_i - q_i)^2)).
    """
    # Uses numpy's efficient vectorized operations
    return np.sqrt(np.sum((point1 - point2)**2)) 

# --- 1. Data Preparation and Splitting ---
def train_test_split_custom(data_df, target_col, test_size=0.3, random_state=42):
    """Splits the DataFrame into training and testing sets."""
    X = data_df.drop(columns=[target_col]).values
    y = data_df[target_col].values
    np.random.seed(random_state)
    test_rows = int(len(X) * test_size)
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    test_indices = indices[:test_rows]
    train_indices = indices[test_rows:]
    X_train, X_test = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]
    return X_train, X_test, y_train, y_test

# --- 2. Finding Neighbors ---
def find_neighbors(X_train, y_train, X_test_point, k):
    """
    Calculates the distance from one test point to all training points,
    sorts them, and returns the k-nearest neighbors' labels.
    """
    distances = []
    for i, x_train_point in enumerate(X_train):
        dist = euclidean_distance(x_train_point, X_test_point)
        distances.append((dist, y_train[i]))
        
    distances.sort(key=lambda x: x[0])
    k_neighbors_labels = [label for dist, label in distances[:k]]
    return k_neighbors_labels

# --- 3. Making a Prediction (Majority Voting) ---
def predict_classification(k_neighbors_labels):
    """
    Determines the final prediction by majority vote among the k neighbors' labels.
    """
    vote_counts = Counter(k_neighbors_labels)
    # Return the most common label
    return vote_counts.most_common(1)[0][0]

# --- 4. The Main k-NN Model Function ---
def knn_predict(X_train, y_train, X_test, k):
    """Applies the k-NN algorithm to a full test set."""
    predictions = []
    for x_test_point in X_test:
        k_neighbors_labels = find_neighbors(X_train, y_train, x_test_point, k)
        prediction = predict_classification(k_neighbors_labels)
        predictions.append(prediction)
    return np.array(predictions)

# --- 5. Accuracy Evaluation ---
def calculate_accuracy(y_true, y_pred):
    """Calculates the classification accuracy."""
    correct_predictions = np.sum(y_true == y_pred)
    total_samples = len(y_true)
    return correct_predictions / total_samples


## --- ADDED: Standardization Function ---

def standardize_data(X_train, X_test):
    """
    Standardizes the data (Z-score normalization).
    X_standardized = (X - mean) / standard deviation
    
    The mean and std are calculated ONLY on the training data.
    """
    # Calculate mean and standard deviation ONLY from the training set
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    
    # Handle columns with zero standard deviation (to avoid division by zero)
    # If std is 0, the data points in that column are identical, so we leave them as 0 after subtraction.
    std[std == 0] = 1.0 
    
    # Apply the transformation to both training and testing sets
    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std
    
    return X_train_scaled, X_test_scaled

# ====================================================================
# --- MODIFIED K-NN EXECUTION BLOCK ---
# ====================================================================

# ... (Using the same df and utility functions from the previous successful run) ...

SAMPLE_SIZE_KNN = min(1000, df.shape[0]) 
df_sample = df.sample(SAMPLE_SIZE_KNN, random_state=42).copy()

k_values_list = [1, 3, 5, 7, 9] 
TARGET_COLUMN = 'Status'
results = [] 

print(f"\n\n{'='*70}\n--- Custom k-NN Model Evaluation AFTER STANDARDIZATION ---")
print(f"Running on a sample of {SAMPLE_SIZE_KNN} rows. Expect better accuracy!")
print(f"K-values to test: {k_values_list}\n{'='*70}")

# 1. Split the data (UNSCALED)
X_train_unscaled, X_test_unscaled, y_train, y_test = train_test_split_custom(
    df_sample, TARGET_COLUMN, test_size=0.3, random_state=42
)

# 2. STANDARDIZE THE DATA
X_train_scaled, X_test_scaled = standardize_data(X_train_unscaled, X_test_unscaled)

print(f"Training set size: {len(X_train_scaled)} | Testing set size: {len(X_test_scaled)}")
print("-" * 70)

# 3. Loop through each K value using SCALED data
for k in k_values_list:
    print(f"Evaluating k = {k}...")
    
    # Make predictions using the custom k-NN function with SCALED data
    y_pred = knn_predict(X_train_scaled, y_train, X_test_scaled, k=k)
    
    # Evaluate the model
    accuracy = calculate_accuracy(y_test, y_pred)
    
    # Store the result
    results.append({'K Value': k, 'Accuracy': accuracy})

# 4. Present Results
df_results = pd.DataFrame(results)

print("\n--- Model Performance Summary by K Value (Standardized Data) ---")
print(df_results.to_string(index=False, float_format="%.4f"))
print("-" * 70)

# Identify the best K value
best_k = df_results.loc[df_results['Accuracy'].idxmax()]

print(f"🏆 **Best K Value Found:** k = {int(best_k['K Value'])} with an Accuracy of {best_k['Accuracy']:.4f}")
print("=" * 70)



--- Custom k-NN Model Evaluation AFTER STANDARDIZATION ---
Running on a sample of 1000 rows. Expect better accuracy!
K-values to test: [1, 3, 5, 7, 9]
Training set size: 700 | Testing set size: 300
----------------------------------------------------------------------
Evaluating k = 1...
Evaluating k = 3...
Evaluating k = 5...
Evaluating k = 7...
Evaluating k = 9...

--- Model Performance Summary by K Value (Standardized Data) ---
 K Value  Accuracy
       1    0.2133
       3    0.2100
       5    0.2033
       7    0.2033
       9    0.2267
----------------------------------------------------------------------
🏆 **Best K Value Found:** k = 9 with an Accuracy of 0.2267


In [7]:
def standardize_data(X_train, X_test):
    """
    Standardizes the data (Z-score normalization).
    X_standardized = (X - mean) / standard deviation
    
    The mean and std are calculated ONLY on the training data 
    and then applied to both training and test sets.
    """
    # Calculate mean and standard deviation ONLY from the training set
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    
    # Handle columns with zero standard deviation (to avoid division by zero)
    # If std is 0, the feature is constant, and the data is left as 0 after subtraction.
    std[std == 0] = 1.0 
    
    # Apply the transformation to both training and testing sets
    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std
    
    return X_train_scaled, X_test_scaled

In [8]:
def calculate_full_metrics(y_true, y_pred, labels):
    """
    Calculates Accuracy, Precision, Recall, F1 Score (weighted),
    Sensitivity, and Specificity.
    """
    
    # 1. Standard Multi-class Metrics (Weighted Averages)
    accuracy = accuracy_score(y_true, y_pred)
    precision_w = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall_w = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_w = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    sensitivity_w = recall_w # Sensitivity is Weighted Recall (TPR)
    
    # 2. Specificity (Weighted Average True Negative Rate - via one-vs-rest)
    tnr_scores = []
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    for i in range(len(labels)):
        # One-vs-Rest calculation for class 'i'
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FN + FP)
        
        # Specificity (True Negative Rate) = TN / (TN + FP)
        tnr = TN / (TN + FP) if (TN + FP) != 0 else 0.0
        
        # Weight by class support (count)
        support = cm[i, :].sum()
        tnr_scores.append(tnr * support)
    
    specificity_w = sum(tnr_scores) / len(y_true)
    
    return {
        'Accuracy': accuracy,
        'Precision': precision_w,
        'Recall': recall_w,
        'Sensitivity': sensitivity_w, 
        'Specificity': specificity_w, 
        'F1 Score': f1_w
    }

In [12]:
# ====================================================================
# --- ADD: MULTI-CLASS METRICS (NO SKLEARN) ---
# ====================================================================

def classification_metrics(y_true, y_pred):
    """
    Computes multi-class Accuracy, Precision, Recall, Sensitivity,
    Specificity, and F1 Score without sklearn.
    Macro-averaging is used for all metrics.
    """

    classes = np.unique(y_true)
    num_classes = len(classes)

    # Initialize counters
    TP = dict.fromkeys(classes, 0)
    FP = dict.fromkeys(classes, 0)
    FN = dict.fromkeys(classes, 0)
    TN = dict.fromkeys(classes, 0)

    # Compute TP, FP, FN, TN for each class
    for c in classes:
        for yt, yp in zip(y_true, y_pred):
            if yt == c and yp == c:
                TP[c] += 1
            elif yt != c and yp == c:
                FP[c] += 1
            elif yt == c and yp != c:
                FN[c] += 1
            elif yt != c and yp != c:
                TN[c] += 1

    # Compute macro metrics
    precision_list = []
    recall_list = []
    specificity_list = []
    f1_list = []

    for c in classes:
        tp, fp, fn, tn = TP[c], FP[c], FN[c], TN[c]

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        precision_list.append(precision)
        recall_list.append(recall)
        specificity_list.append(specificity)
        f1_list.append(f1)

    accuracy = np.sum(y_true == y_pred) / len(y_true)
    macro_precision = np.mean(precision_list)
    macro_recall = np.mean(recall_list)
    macro_specificity = np.mean(specificity_list)
    macro_f1 = np.mean(f1_list)

    return accuracy, macro_precision, macro_recall, macro_specificity, macro_f1


# ====================================================================
# --- HOLD-OUT VALIDATION (WITH METRICS) ---
# ====================================================================
print("\n\n================ HOLD-OUT VALIDATION (70/30) ================")

holdout_results = []

for k in k_values_list:
    print(f"\nTesting k = {k}")

    y_pred = knn_predict(X_train_scaled, y_train, X_test_scaled, k=k)

    acc, prec, rec, spec, f1 = classification_metrics(y_test, y_pred)

    holdout_results.append({
        "K": k,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Sensitivity": rec,
        "Specificity": spec,
        "F1 Score": f1
    })

df_holdout = pd.DataFrame(holdout_results)
print(df_holdout.to_string(index=False))


# ====================================================================
# --- CUSTOM 10-FOLD CROSS VALIDATION (NO SKLEARN) ---
# ====================================================================
print("\n\n================ 10-FOLD CROSS VALIDATION (CUSTOM) ================")

df_cv = df_sample.copy().reset_index(drop=True)
fold_size = len(df_cv) // 10

cv_results = []

for k in k_values_list:
    fold_metrics = []

    for fold in range(10):
        print(f"Evaluating Fold {fold+1} for k = {k}")

        # Create fold splits
        start = fold * fold_size
        end = start + fold_size

        df_test_fold = df_cv.iloc[start:end]
        df_train_fold = pd.concat([df_cv.iloc[:start], df_cv.iloc[end:]])

        X_train_f = df_train_fold.drop(columns=[TARGET_COLUMN]).values
        y_train_f = df_train_fold[TARGET_COLUMN].values
        X_test_f = df_test_fold.drop(columns=[TARGET_COLUMN]).values
        y_test_f = df_test_fold[TARGET_COLUMN].values

        # Standardize
        X_train_f, X_test_f = standardize_data(X_train_f, X_test_f)

        # Predict
        y_pred_f = knn_predict(X_train_f, y_train_f, X_test_f, k=k)

        # Compute metrics
        metrics = classification_metrics(y_test_f, y_pred_f)
        fold_metrics.append(metrics)

    # Average over 10 folds
    metrics_avg = np.mean(fold_metrics, axis=0)

    cv_results.append({
        "K": k,
        "Accuracy": metrics_avg[0],
        "Precision": metrics_avg[1],
        "Recall": metrics_avg[2],
        "Sensitivity": metrics_avg[2],
        "Specificity": metrics_avg[3],
        "F1 Score": metrics_avg[4]
    })

df_cv_results = pd.DataFrame(cv_results)

print("\n\n===== FINAL 10-FOLD CROSS-VALIDATION RESULTS =====")
print(df_cv_results.to_string(index=False))




================ HOLD-OUT VALIDATION (70/30) ================

Testing k = 1

Testing k = 3

Testing k = 5

Testing k = 7

Testing k = 9
 K  Accuracy  Precision   Recall  Sensitivity  Specificity  F1 Score
 1  0.213333   0.229666 0.218582     0.218582     0.884478  0.218101
 3  0.210000   0.216291 0.211218     0.211218     0.884208  0.208026
 5  0.203333   0.170146 0.166248     0.166248     0.882337  0.163177
 7  0.203333   0.172795 0.170960     0.170960     0.882520  0.167318
 9  0.226667   0.183628 0.176353     0.176353     0.885921  0.176690


================ 10-FOLD CROSS VALIDATION (CUSTOM) ================
Evaluating Fold 1 for k = 1
Evaluating Fold 2 for k = 1
Evaluating Fold 3 for k = 1
Evaluating Fold 4 for k = 1
Evaluating Fold 5 for k = 1
Evaluating Fold 6 for k = 1
Evaluating Fold 7 for k = 1
Evaluating Fold 8 for k = 1
Evaluating Fold 9 for k = 1
Evaluating Fold 10 for k = 1
Evaluating Fold 1 for k = 3
Evaluating Fold 2 for k = 3
Evaluating Fold 3 for k = 3
Evaluating F